# Torso Decompositions (ESA SpOC-3) – Metaheuristika

Detyra 2, Problemi 4 – Inteligjenca Artificiale (Master), Universiteti i Prishtinës.

* **Problemi** ([README zyrtar](https://github.com/esa/SpOC3/tree/main/Challenge%201%20Torso%20Decompositions)):
  vektori `x = [p_1..p_n, t]` (permutacion + prag). Torso = n − t nyjet e fundit. Objektivat: madhësia e
  torsos ↑ dhe gjerësia e torsos (outdegree max pas shtimit të brinjëve) ↓. Dorëzohen ≤ 20 vektorë,
  vlerësimi bëhet me hypervolume.
* **Zgjidhja:** renditje konstruktive (min-degree, min-fill) → **Simulated Annealing** dhe
  **Iterated Local Search** me lëvizjet swap/insert. Objektivi = hypervolume i frontit të permutacionit.
  Në fund bashkohen frontet dhe zgjidhen 20 pikat me HV maksimal (programim dinamik).
* Çdo ekzekutim zgjat **900 s**, me **2 seed** për algoritëm, paralelisht në të gjitha CPU-të e Kaggle.

In [ ]:
import os, glob, urllib.request
DATA = "data"
os.makedirs(DATA, exist_ok=True)
for name in ("easy", "medium", "hard"):
    found = glob.glob(f"/kaggle/input/**/{name}.gr", recursive=True)
    dst = os.path.join(DATA, f"{name}.gr")
    if found:
        os.system(f"cp '{found[0]}' '{dst}'")
    elif not os.path.exists(dst):
        url = f"https://raw.githubusercontent.com/esa/SpOC3/main/Challenge%201%20Torso%20Decompositions/data/{name}.gr"
        urllib.request.urlretrieve(url, dst)
    print(name, os.path.getsize(dst), "bytes")
print("CPU:", os.cpu_count())

### `torso.py`

In [ ]:
%%writefile torso.py
"""
ESA SpOC-3, Challenge 1 – Torso Decompositions: grafi, vlerësimi, fronti Pareto dhe hypervolume.

Përkufizimet (README zyrtar, https://github.com/esa/SpOC3):
* Vektori i vendimit x = [p_1, ..., p_n, t]: permutacion i nyjeve 0..n-1 dhe pragu t ∈ {0..n-1}.
* Brinja {u, v} orientohet (u, v) nëse u del para v në permutacion.
* Skanohet permutacioni nga e majta në të djathtë; kur nyja u ka dy brinjë dalëse drejt v dhe w që
  nuk janë fqinjë, shtohet brinja {v, w} (e orientuar sipas permutacionit).
* Torso = n - t elementet e fundit; madhësia = n - t; gjerësia = outdegree maksimal (me brinjët e reja)
  i një nyjeje në torso.
* Objektivat: madhësia max (t min) dhe gjerësia min. Gjerësia > 500 vlerësohet si 501. Deri 20 vektorë
  për dorëzim, cilësia matet me hypervolume.

Vlerësimi efikas
----------------
Punojmë në "hapësirën e pozicioneve": out[i] = bitset i pozicioneve j > i që janë fqinjë me nyjen në
pozicionin i. Procedura zyrtare e shtimit të brinjëve është "loja e eliminimit": kur përpunohet
pozicioni i, fqinjët e tij të mëvonshëm bëhen klikë. Mjafton t'ia kalojmë bashkësinë out[i] fqinjit më
të hershëm v (out[v] |= out[i] \\ {v}); kur të përpunohet v, klika përhapet më tej. Kjo jep të njëjtin
graf të mbushur (Rose, Tarjan & Lueker, 1976) dhe outdegree[i] = |out[i]|. Kosto: O(n) operacione mbi
bitset-e, pra disa milisekonda edhe për grafin hard.
"""

from __future__ import annotations

import os

WIDTH_CAP = 501          # gjerësia > 500 vlerësohet si 501 (rregullat zyrtare)
MAX_POINTS = 20          # maksimumi i vektorëve në një dorëzim

PROBLEM_IDS = {"easy": "small-graph", "medium": "medium-graph", "hard": "large-graph"}
CHALLENGE_ID = "spoc-3-torso-decompositions"


class Graph:
    def __init__(self, n: int, edges: list[tuple[int, int]], name: str = "graph"):
        self.n = n
        self.name = name
        self.edges = edges
        self.adj = [0] * n                    # bitset i fqinjëve (në hapësirën e nyjeve)
        self.nbrs = [[] for _ in range(n)]
        for u, v in edges:
            if not (self.adj[u] >> v) & 1:
                self.adj[u] |= 1 << v
                self.adj[v] |= 1 << u
                self.nbrs[u].append(v)
                self.nbrs[v].append(u)
        self.m = sum(len(a) for a in self.nbrs) // 2


def load_graph(path: str) -> Graph:
    edges, n = [], 0
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if len(parts) >= 2:
                u, v = int(parts[0]), int(parts[1])
                if u != v:
                    edges.append((u, v))
                    n = max(n, u + 1, v + 1)
    return Graph(n, edges, os.path.splitext(os.path.basename(path))[0])


# ============================================================================ vlerësimi

def outdegrees(g: Graph, perm: list[int]) -> list[int]:
    """outdeg[i] = outdegree (me brinjët e shtuara) i nyjes në pozicionin i të permutacionit."""
    n = g.n
    pos = [0] * n
    for i, v in enumerate(perm):
        pos[v] = i
    out = [0] * n
    for i, v in enumerate(perm):
        m = 0
        for u in g.nbrs[v]:
            j = pos[u]
            if j > i:
                m |= 1 << j
        out[i] = m
    deg = [0] * n
    for i in range(n):
        m = out[i]
        if m:
            low = m & -m                       # fqinji i mëvonshëm më i hershëm
            v = low.bit_length() - 1
            out[v] |= m ^ low
            deg[i] = m.bit_count()
    return deg


def suffix_widths(deg: list[int]) -> list[int]:
    """w[t] = gjerësia e torsos kur pragu është t = max(deg[t:]), e kufizuar në 501."""
    n = len(deg)
    w = [0] * n
    best = 0
    for i in range(n - 1, -1, -1):
        if deg[i] > best:
            best = deg[i]
        w[i] = min(best, WIDTH_CAP)
    return w


def evaluate(g: Graph, x: list[int]) -> tuple[int, int]:
    """Vlerësimi i një vektori x = [p_1..p_n, t] → (madhësia e torsos, gjerësia e torsos)."""
    perm, t = list(x[:-1]), int(x[-1])
    assert sorted(perm) == list(range(g.n)), "pjesa e parë e x duhet të jetë permutacion i 0..n-1"
    assert 0 <= t <= g.n - 1, "t duhet të jetë në {0..n-1}"
    return g.n - t, suffix_widths(outdegrees(g, perm))[t]


# ============================================================================ fronti dhe hypervolume
# Në hapësirën e minimizimit përdorim (t, gjerësia); pika e referencës = (n, 501).

def front_from_widths(w: list[int]) -> list[tuple[int, int]]:
    """Pikat jo-dominuese (t, gjerësia) që jep një permutacion: për çdo gjerësi, t më i vogël."""
    pts, last = [], None
    for t, wt in enumerate(w):
        if wt != last and wt < WIDTH_CAP:
            pts.append((t, wt))
            last = wt
    return pts


def staircase_hv(w: list[int]) -> int:
    """Hypervolume i frontit të plotë të një permutacioni = Σ_t (501 - w[t]).
    Përdoret si funksion objektiv nga metaheuristikat (maksimizohet)."""
    return sum(WIDTH_CAP - x for x in w)


def hypervolume(points: list[tuple[int, int]], n: int) -> int:
    """Hypervolume 2D (minimizim i t dhe gjerësisë) me pikë reference (n, 501)."""
    pts = pareto_filter(points)
    hv = 0
    for k, (t, wt) in enumerate(pts):
        t_next = pts[k + 1][0] if k + 1 < len(pts) else n
        hv += (t_next - t) * (WIDTH_CAP - wt)
    return hv


def pareto_filter(points):
    """Pikat jo-dominuese, të renditura sipas t rritës (gjerësia zbret)."""
    best = {}
    for t, wt in points:
        if wt < WIDTH_CAP and (wt not in best or t < best[wt]):
            best[wt] = t
    pts = sorted((t, wt) for wt, t in best.items())
    out, min_w = [], None
    for t, wt in pts:
        if min_w is None or wt < min_w:
            out.append((t, wt))
            min_w = wt
    return out


def select_best_subset(points, n: int, k: int = MAX_POINTS):
    """Zgjedh ≤ k pika që maksimizojnë hypervolume-in (programim dinamik, optimal në 2D)."""
    pts = pareto_filter(points)
    m = len(pts)
    if m <= k:
        return pts
    # best[c][i] = HV maksimal duke filluar me pikën i dhe duke përdorur c pika nga i..m-1
    NEG = -1
    best = [[NEG] * m for _ in range(k + 1)]
    choice = [[-1] * m for _ in range(k + 1)]
    for i in range(m):
        best[1][i] = (n - pts[i][0]) * (WIDTH_CAP - pts[i][1])
    for c in range(2, k + 1):
        for i in range(m - c, -1, -1):
            ti, hi = pts[i][0], WIDTH_CAP - pts[i][1]
            b, bj = NEG, -1
            for j in range(i + 1, m - c + 2):
                if best[c - 1][j] < 0:
                    continue
                val = (pts[j][0] - ti) * hi + best[c - 1][j]
                if val > b:
                    b, bj = val, j
            best[c][i], choice[c][i] = b, bj
    i = max(range(m), key=lambda i: best[k][i])
    out, c = [], k
    while i >= 0 and c >= 1:
        out.append(pts[i])
        i, c = choice[c][i], c - 1
    return out

### `search.py`

In [ ]:
%%writefile search.py
"""
Zgjidhjet fillestare (konstruktive) dhe metaheuristikat për Torso Decompositions.

* Evaluator – mban permutacionin dhe fqinjësinë në "hapësirën e pozicioneve" si bitset-e, që lëvizjet
  swap/insert të përditësohen në mënyrë inkrementale (pa e rindërtuar nga lista e brinjëve).
* Konstruktive: min-degree dhe min-fill (renditje eliminimi klasike për treewidth).
* Metaheuristikat: Simulated Annealing (SA) dhe Iterated Local Search (ILS).
  Funksioni objektiv (maksimizohet) = hypervolume i frontit të plotë që jep permutacioni,
  HV(π) = Σ_t (501 − w_π(t)), ku w_π(t) = gjerësia e torsos me prag t.
"""

from __future__ import annotations

import math
import random
import time

from torso import WIDTH_CAP, Graph, front_from_widths, suffix_widths


# ============================================================================ vlerësues inkremental

class Evaluator:
    def __init__(self, g: Graph, perm: list[int]):
        self.g = g
        n = self.n = g.n
        self.perm = list(perm)
        self.pos = [0] * n
        for i, v in enumerate(self.perm):
            self.pos[v] = i
        # P[i] = bitset i pozicioneve të fqinjëve të nyjes në pozicionin i
        self.P = [0] * n
        for i, v in enumerate(self.perm):
            m = 0
            for u in g.nbrs[v]:
                m |= 1 << self.pos[u]
            self.P[i] = m
        full = (1 << n) - 1
        self.later = [full ^ ((1 << (i + 1)) - 1) for i in range(n)]   # bitet j > i
        self.evals = 0

    def widths(self) -> list[int]:
        """Gjerësia e torsos për çdo prag t (loja e eliminimit, O(n) operacione bitset)."""
        self.evals += 1
        n, P, later = self.n, self.P, self.later
        out = [P[i] & later[i] for i in range(n)]
        deg = [0] * n
        for i in range(n):
            m = out[i]
            if m:
                low = m & -m
                out[low.bit_length() - 1] |= m ^ low
                deg[i] = m.bit_count()
        self.deg = deg
        return suffix_widths(deg)

    # ---------------------------------------------------------------- lëvizjet
    def swap(self, a: int, b: int) -> None:
        """Ndërron nyjet në pozicionet a dhe b."""
        if a == b:
            return
        na, nb = self.perm[a], self.perm[b]
        P, pos, mask = self.P, self.pos, (1 << a) | (1 << b)
        for u in set(self.g.nbrs[na]) | set(self.g.nbrs[nb]):
            k = pos[u]
            r = P[k]
            if ((r >> a) ^ (r >> b)) & 1:          # vetëm njëri nga bitet është 1
                P[k] = r ^ mask
        P[a], P[b] = P[b], P[a]
        self.perm[a], self.perm[b] = nb, na
        pos[na], pos[nb] = b, a

    def insert(self, i: int, j: int) -> None:
        """Heq nyjen nga pozicioni i dhe e vendos në pozicionin j (të tjerat zhvendosen)."""
        if i == j:
            return
        lo, hi = min(i, j), max(i, j)
        seg = ((1 << (hi - lo + 1)) - 1) << lo
        P = self.P
        if i < j:      # biti i -> j, bitet i+1..j -> një poshtë
            for k in range(self.n):
                r = P[k]
                s = r & seg
                if s:
                    P[k] = (r ^ s) | ((s >> 1) & seg) | (((s >> i) & 1) << j)
        else:          # biti i -> j (j < i), bitet j..i-1 -> një lart
            for k in range(self.n):
                r = P[k]
                s = r & seg
                if s:
                    P[k] = (r ^ s) | ((s << 1) & seg) | (((s >> i) & 1) << j)
        P.insert(j, P.pop(i))
        self.perm.insert(j, self.perm.pop(i))
        for k in range(lo, hi + 1):
            self.pos[self.perm[k]] = k


def hv_of(w: list[int]) -> int:
    return sum(WIDTH_CAP - x for x in w)


# ============================================================================ konstruktive

def min_degree_order(g: Graph, rng: random.Random | None = None) -> list[int]:
    """Eliminon gjithmonë nyjen me shkallën më të vogël në grafin aktual (me brinjët e shtuara)."""
    adj = list(g.adj)
    alive = (1 << g.n) - 1
    deg = [a.bit_count() for a in adj]
    order = []
    remaining = set(range(g.n))
    for _ in range(g.n):
        dmin = min(deg[v] for v in remaining)
        cands = [v for v in remaining if deg[v] == dmin]
        v = rng.choice(cands) if rng else cands[0]
        order.append(v)
        remaining.discard(v)
        alive &= ~(1 << v)
        nb = adj[v] & alive
        m = nb
        while m:                                  # fqinjët bëhen klikë
            low = m & -m
            u = low.bit_length() - 1
            m ^= low
            adj[u] = (adj[u] | nb) & ~low & alive
            deg[u] = adj[u].bit_count()
    return order


def min_fill_order(g: Graph, rng: random.Random | None = None) -> list[int]:
    """Eliminon nyjen që shton më pak brinjë të reja (fill-in). Më e shtrenjtë se min-degree."""
    adj = list(g.adj)
    alive = (1 << g.n) - 1
    remaining = set(range(g.n))

    def fill(v):
        nb = adj[v] & alive
        missing, m = 0, nb
        while m:
            low = m & -m
            u = low.bit_length() - 1
            m ^= low
            missing += (nb & ~adj[u] & ~low).bit_count()
        return missing // 2

    score = {v: fill(v) for v in remaining}
    order = []
    for _ in range(g.n):
        best = min(score[v] for v in remaining)
        cands = [v for v in remaining if score[v] == best]
        v = rng.choice(cands) if rng else min(cands, key=lambda c: adj[c].bit_count())
        order.append(v)
        remaining.discard(v)
        alive &= ~(1 << v)
        nb = adj[v] & alive
        m = nb
        while m:
            low = m & -m
            u = low.bit_length() - 1
            m ^= low
            adj[u] = (adj[u] | nb) & ~low & alive
        # fill ndryshon vetëm për nyjet në distancë ≤ 2 nga v
        touched, m = nb, nb
        while m:
            low = m & -m
            m ^= low
            touched |= adj[low.bit_length() - 1] & alive
        m = touched
        while m:
            low = m & -m
            m ^= low
            score[low.bit_length() - 1] = fill(low.bit_length() - 1)
        score.pop(v, None)
    return order


# ============================================================================ metaheuristikat

def random_move(ev: Evaluator, rng: random.Random, w=None, window: int = 30):
    """Gjeneron një lëvizje dhe lëvizjen e kundërt (për undo).

    * 50%: lëvizje lokale – j zgjidhet në dritaren |i − j| ≤ window (ndryshim i vogël i renditjes);
    * 30%: lëvizje mbi një nyje kritike (deg[i] = w[i], d.m.th. nyja që përcakton gjerësinë e torsos
      për një prag) – zhvendoset brenda dritares;
    * 20%: swap/insert mes dy pozicioneve krejt të rastit (diversifikim).
    """
    n = ev.n
    r = rng.random()
    if r < 0.8:
        if r >= 0.5 and w is not None and getattr(ev, "deg", None) is not None:
            crit = [i for i in range(n - 1) if ev.deg[i] == w[i] and ev.deg[i] > 0]
            a = rng.choice(crit) if crit else rng.randrange(n)
        else:
            a = rng.randrange(n)
        b = min(n - 1, max(0, a + rng.randint(-window, window)))
        if b == a:
            b = a + 1 if a + 1 < n else a - 1
    else:
        a, b = rng.randrange(n), rng.randrange(n)
        while b == a:
            b = rng.randrange(n)
    if rng.random() < 0.5:
        return ("swap", a, b), ("swap", a, b)
    return ("insert", a, b), ("insert", b, a)


def apply(ev: Evaluator, move):
    if move[0] == "swap":
        ev.swap(move[1], move[2])
    else:
        ev.insert(move[1], move[2])


class Archive:
    """Ruaj pikat jo-dominuese (t, gjerësia) dhe permutacionin që i jep secilën."""

    def __init__(self):
        self.best = {}                                 # gjerësia -> (t, perm)

    def add(self, w, perm) -> None:
        for t, wt in front_from_widths(w):
            if wt not in self.best or t < self.best[wt][0]:
                self.best[wt] = (t, list(perm))

    def points(self):
        return [(t, wt) for wt, (t, _) in self.best.items()]


def calibrate_t0(ev, rng, samples=100):
    """T0 = mesatarja e |Δ| për lëvizje të rastit që e përkeqësojnë zgjidhjen: në fillim një
    përkeqësim "tipik" pranohet me probabilitet e^-1 ≈ 0.37."""
    w = ev.widths()
    base, deltas = hv_of(w), []
    for _ in range(samples):
        move, undo = random_move(ev, rng, w)
        apply(ev, move)
        d = hv_of(ev.widths()) - base
        apply(ev, undo)
        if d < 0:
            deltas.append(-d)
    ev.widths()
    return sum(deltas) / len(deltas) if deltas else 1.0


def simulated_annealing(g, start, seconds, seed, log_every=1.0):
    """SA: pranon përkeqësime me probabilitet exp(Δ/T); T zvogëlohet gjeometrikisht me kohën
    nga T0 (i kalibruar) deri në 0.01."""
    rng = random.Random(seed)
    ev = Evaluator(g, start)
    w = ev.widths()
    cur, cur_w, cur_deg = hv_of(w), w, ev.deg
    best, best_perm = cur, list(ev.perm)
    archive = Archive()
    archive.add(w, ev.perm)
    T0 = max(1.0, calibrate_t0(ev, rng))
    cur_deg = ev.deg
    T_end = 0.01
    start_t = time.perf_counter()
    history = [(0.0, ev.evals, best)]
    next_log = log_every
    while True:
        el = time.perf_counter() - start_t
        if el >= seconds:
            break
        T = T0 * (T_end / T0) ** (el / seconds)
        move, undo = random_move(ev, rng, cur_w)
        apply(ev, move)
        w = ev.widths()
        val = hv_of(w)
        d = val - cur
        if d >= 0 or rng.random() < math.exp(d / T):
            cur, cur_w, cur_deg = val, w, ev.deg
            archive.add(w, ev.perm)
            if cur > best:
                best, best_perm = cur, list(ev.perm)
        else:
            apply(ev, undo)
            ev.deg = cur_deg
        if el >= next_log:
            history.append((el, ev.evals, best))
            next_log += log_every
    history.append((time.perf_counter() - start_t, ev.evals, best))
    return {"best_hv": best, "best_perm": best_perm, "archive": archive, "history": history, "evals": ev.evals}


def iterated_local_search(g, start, seconds, seed, ls_tries=200, kick=8, log_every=1.0):
    """ILS: hill climbing (first improvement me lëvizje të rastit) deri në `ls_tries` dështime
    radhazi, pastaj perturbim me `kick` lëvizje të rastit; pranohet nëse s'është më keq se më i miri."""
    rng = random.Random(seed)
    ev = Evaluator(g, start)
    w = ev.widths()
    cur, cur_w, cur_deg = hv_of(w), w, ev.deg
    best, best_perm = cur, list(ev.perm)
    archive = Archive()
    archive.add(w, ev.perm)
    start_t = time.perf_counter()
    history = [(0.0, ev.evals, best)]
    next_log = log_every
    while time.perf_counter() - start_t < seconds:
        # --- kërkim lokal
        fails = 0
        while fails < ls_tries and time.perf_counter() - start_t < seconds:
            move, undo = random_move(ev, rng, cur_w)
            apply(ev, move)
            w = ev.widths()
            val = hv_of(w)
            if val > cur:
                cur, cur_w, cur_deg, fails = val, w, ev.deg, 0
                archive.add(w, ev.perm)
            elif val == cur:                           # lëvizje anësore (plateau), lejohet
                cur_w, cur_deg = w, ev.deg
                fails += 1
                archive.add(w, ev.perm)
            else:
                apply(ev, undo)
                ev.deg = cur_deg
                fails += 1
            el = time.perf_counter() - start_t
            if el >= next_log:
                history.append((el, ev.evals, max(best, cur)))
                next_log += log_every
        # --- pranimi
        if cur >= best:
            best, best_perm = cur, list(ev.perm)
        else:                                          # kthehu te zgjidhja më e mirë
            ev = _restart(ev, g, best_perm)
            cur = best
        # --- perturbimi
        for _ in range(kick):
            apply(ev, random_move(ev, rng)[0])
        w = ev.widths()
        cur, cur_w, cur_deg = hv_of(w), w, ev.deg
        archive.add(w, ev.perm)
    history.append((time.perf_counter() - start_t, ev.evals, best))
    return {"best_hv": best, "best_perm": best_perm, "archive": archive, "history": history, "evals": ev.evals}


def _restart(ev, g, perm):
    evals = ev.evals
    new = Evaluator(g, perm)
    new.evals = evals
    return new

### `run.py`

In [ ]:
%%writefile run.py
"""
Ekzekutimi i plotë për Torso Decompositions (përdoret lokalisht dhe në Kaggle).

    python run.py --seconds 30                 # provë e shpejtë lokale
    python run.py --seconds 900 --seeds 2      # versioni i Kaggle (paralel në të gjitha CPU-të)

Hapat:
1. zgjidhjet konstruktive (min-degree, min-fill) për çdo instancë;
2. SA dhe ILS për çdo instancë dhe çdo seed, paralelisht (multiprocessing);
3. bashkimi i fronteve, zgjedhja e ≤ 20 pikave me hypervolume maksimal (DP);
4. verifikimi i çdo vektori me vlerësimin e pavarur `torso.evaluate`;
5. dalja: submissions/*.json, results/results.md, results/*.png, results/summary.json.
"""

from __future__ import annotations

import argparse
import json
import multiprocessing as mp
import os
import random
import time

from search import Evaluator, hv_of, iterated_local_search, min_degree_order, min_fill_order, simulated_annealing
from torso import (CHALLENGE_ID, PROBLEM_IDS, WIDTH_CAP, evaluate, front_from_widths, hypervolume,
                   load_graph, pareto_filter, select_best_subset)

INSTANCES = ["easy", "medium", "hard"]
ALGOS = {"SA": simulated_annealing, "ILS": iterated_local_search}


def constructive(name, data_dir):
    g = load_graph(os.path.join(data_dir, f"{name}.gr"))
    out = {}
    t = time.perf_counter()
    out["min-degree"] = (min_degree_order(g), time.perf_counter() - t)
    if g.m < 50_000:                      # min-fill është shumë i shtrenjtë për grafin hard
        t = time.perf_counter()
        out["min-fill"] = (min_fill_order(g), time.perf_counter() - t)
    return out


def task(args):
    name, algo, seed, start, seconds, data_dir = args
    g = load_graph(os.path.join(data_dir, f"{name}.gr"))
    r = ALGOS[algo](g, start, seconds, seed)
    pts = pareto_filter(r["archive"].points())
    keep = {wt: r["archive"].best[wt] for _, wt in pts}          # vetëm pikat jo-dominuese
    return {"instance": name, "algo": algo, "seed": seed, "best_hv": r["best_hv"], "evals": r["evals"],
            "history": r["history"], "archive": keep,
            "hv20": hypervolume(select_best_subset(pts, g.n), g.n), "n_points": len(pts)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--seconds", type=float, default=30)
    ap.add_argument("--seeds", type=int, default=1)
    ap.add_argument("--data", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "data"))
    ap.add_argument("--out", default=os.path.dirname(os.path.abspath(__file__)))
    ap.add_argument("--workers", type=int, default=os.cpu_count())
    args = ap.parse_args()
    res_dir, sub_dir = os.path.join(args.out, "results"), os.path.join(args.out, "submissions")
    os.makedirs(res_dir, exist_ok=True)
    os.makedirs(sub_dir, exist_ok=True)
    t_all = time.perf_counter()

    # 1. konstruktive
    graphs, starts, cons_rows = {}, {}, []
    for name in INSTANCES:
        g = graphs[name] = load_graph(os.path.join(args.data, f"{name}.gr"))
        cons = constructive(name, args.data)
        for cname, (perm, secs) in cons.items():
            w = Evaluator(g, perm).widths()
            pts = front_from_widths(w)
            cons_rows.append({"instance": name, "method": cname, "seconds": round(secs, 2),
                              "width_t0": w[0], "staircase_hv": hv_of(w),
                              "hv20": hypervolume(select_best_subset(pts, g.n), g.n), "perm": perm, "w": w})
        best = max((r for r in cons_rows if r["instance"] == name), key=lambda r: r["staircase_hv"])
        starts[name] = best["perm"]
        print(f"[{name}] n={g.n} m={g.m} | " + ", ".join(
            f"{r['method']}: w(t=0)={r['width_t0']} HV={r['staircase_hv']}" for r in cons_rows if r["instance"] == name),
            flush=True)

    # 2. metaheuristikat (paralel)
    tasks = [(name, algo, seed, starts[name], args.seconds, args.data)
             for name in INSTANCES for algo in ALGOS for seed in range(1, args.seeds + 1)]
    print(f"{len(tasks)} ekzekutime × {args.seconds:.0f} s në {args.workers} procese ...", flush=True)
    with mp.Pool(min(args.workers, len(tasks))) as pool:
        runs = pool.map(task, tasks)
    for r in runs:
        print(f"[{r['instance']}] {r['algo']} seed={r['seed']}: staircase HV {r['history'][0][2]} -> {r['best_hv']}, "
              f"HV20 = {r['hv20']}, evals = {r['evals']}", flush=True)

    # 3–4. bashkimi, zgjedhja e 20 pikave, verifikimi, dorëzimi
    summary = {}
    for name in INSTANCES:
        g = graphs[name]
        pool_pts = {}                                          # gjerësia -> (t, perm)
        sources = [(r["archive"]) for r in runs if r["instance"] == name]
        for c in cons_rows:
            if c["instance"] == name:
                sources.append({wt: (t, c["perm"]) for t, wt in front_from_widths(c["w"])})
        for src in sources:
            for wt, (t, perm) in src.items():
                if wt not in pool_pts or t < pool_pts[wt][0]:
                    pool_pts[wt] = (t, perm)
        chosen = select_best_subset([(t, wt) for wt, (t, _) in pool_pts.items()], g.n)
        vectors = []
        for t, wt in chosen:
            x = list(pool_pts[wt][1]) + [t]
            size, width = evaluate(g, x)                       # vlerësim i pavarur
            assert (size, width) == (g.n - t, wt), "verifikimi dështoi"
            vectors.append(x)
        hv = hypervolume(chosen, g.n)
        summary[name] = {"n": g.n, "m": g.m, "hv": hv, "hv_normalized": hv / (g.n * WIDTH_CAP),
                         "points": [{"torso_size": g.n - t, "torso_width": wt} for t, wt in chosen]}
        with open(os.path.join(sub_dir, f"{name}_{PROBLEM_IDS[name]}.json"), "w") as fh:
            json.dump([{"decisionVector": vectors, "problem": PROBLEM_IDS[name], "challenge": CHALLENGE_ID,
                        "name": f"torso-{name}", "description": "SA + ILS, Detyra 2 AI (UP)"}], fh)
        print(f"[{name}] fronti final: {len(chosen)} pika, HV = {hv} (normalizuar {hv / (g.n * WIDTH_CAP):.4f})",
              flush=True)

    with open(os.path.join(res_dir, "summary.json"), "w") as fh:
        json.dump({"seconds_per_run": args.seconds, "seeds": args.seeds, "instances": summary,
                   "constructive": [{k: v for k, v in c.items() if k not in ("perm", "w")} for c in cons_rows],
                   "runs": [{k: v for k, v in r.items() if k != "archive"} for r in runs],
                   "total_seconds": time.perf_counter() - t_all}, fh)
    write_report(res_dir, args, graphs, cons_rows, runs, summary)
    plots(res_dir, graphs, cons_rows, runs, summary)
    print(f"Gjithsej {time.perf_counter() - t_all:.0f} s. Rezultatet në {res_dir}, dorëzimet në {sub_dir}")


def write_report(res_dir, args, graphs, cons_rows, runs, summary):
    md = ["# Rezultatet – Torso Decompositions (ESA SpOC-3)", "",
          f"{args.seconds:.0f} s për ekzekutim, {args.seeds} seed për algoritëm. Hypervolume: minimizim i (t, gjerësia), "
          f"pika e referencës (n, 501). *Staircase HV* = HV i frontit të plotë të një permutacioni (funksioni "
          f"objektiv); *HV20* = HV i 20 pikave më të mira (sa lejon dorëzimi).", "",
          "## Instancat dhe zgjidhjet konstruktive", "",
          "| instanca | n | m | metoda | koha (s) | gjerësia (t=0) | staircase HV | HV20 |", "|---|---|---|---|---|---|---|---|"]
    for c in cons_rows:
        g = graphs[c["instance"]]
        md.append(f"| {c['instance']} | {g.n} | {g.m} | {c['method']} | {c['seconds']} | {c['width_t0']} | "
                  f"{c['staircase_hv']} | {c['hv20']} |")
    md += ["", "## Metaheuristikat", "",
           "| instanca | algoritmi | seed | vlerësime | staircase HV fillim → fund | HV20 | pika jo-dominuese |",
           "|---|---|---|---|---|---|---|"]
    for r in runs:
        md.append(f"| {r['instance']} | {r['algo']} | {r['seed']} | {r['evals']} | {r['history'][0][2]} → {r['best_hv']} | "
                  f"{r['hv20']} | {r['n_points']} |")
    md += ["", "## Fronti final (bashkimi i të gjitha ekzekutimeve, 20 pikat me HV maksimal)", "",
           "| instanca | HV | HV i normalizuar | pikat (madhësia, gjerësia) |", "|---|---|---|---|"]
    for name, s in summary.items():
        pts = ", ".join(f"({p['torso_size']}, {p['torso_width']})" for p in s["points"])
        md.append(f"| {name} | {s['hv']} | {s['hv_normalized']:.4f} | {pts} |")
    md += ["", "![Konvergjenca](convergence.png)", "", "![Frontet Pareto](pareto_fronts.png)", ""]
    with open(os.path.join(res_dir, "results.md"), "w", encoding="utf-8") as fh:
        fh.write("\n".join(md))


def plots(res_dir, graphs, cons_rows, runs, summary):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    colors = {"SA": "#2a6fdb", "ILS": "#e07b1f"}
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, name in zip(axes, INSTANCES):
        for r in runs:
            if r["instance"] == name:
                ts = [h[0] for h in r["history"]]
                hv = [h[2] for h in r["history"]]
                ax.plot(ts, hv, color=colors[r["algo"]], alpha=0.85, label=f"{r['algo']} seed {r['seed']}")
        ax.set_title(f"{name} (n={graphs[name].n})")
        ax.set_xlabel("koha (s)")
        ax.set_ylabel("staircase HV (më i miri)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle("Konvergjenca e SA dhe ILS")
    fig.tight_layout()
    fig.savefig(os.path.join(res_dir, "convergence.png"), dpi=130)
    plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, name in zip(axes, INSTANCES):
        g = graphs[name]
        for c in cons_rows:
            if c["instance"] == name:
                pts = front_from_widths(c["w"])
                ax.step([g.n - t for t, _ in pts], [w for _, w in pts], where="post", alpha=0.6,
                        label=f"{c['method']}")
        fin = summary[name]["points"]
        ax.plot([p["torso_size"] for p in fin], [p["torso_width"] for p in fin], "o", color="crimson",
                label="fronti final (20 pika)")
        ax.set_title(f"{name}: HV = {summary[name]['hv_normalized']:.4f}")
        ax.set_xlabel("madhësia e torsos (n − t)")
        ax.set_ylabel("gjerësia e torsos")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.suptitle("Frontet Pareto (madhësia ↑, gjerësia ↓)")
    fig.tight_layout()
    fig.savefig(os.path.join(res_dir, "pareto_fronts.png"), dpi=130)
    plt.close(fig)


if __name__ == "__main__":
    main()

### Testet (vlerësimi kundrejt shembullit zyrtar dhe procedurës fjalë për fjalë)

In [ ]:
%%writefile test_torso.py
"""Teste: python -m unittest test_torso -v"""

import itertools
import random
import unittest

from torso import (Graph, evaluate, front_from_widths, hypervolume, outdegrees, select_best_subset,
                   staircase_hv, suffix_widths)

README_EDGES = [(0, 1), (0, 2), (1, 6), (2, 8), (3, 8), (3, 9), (4, 5), (4, 6), (4, 7)]


def literal_outdegrees(g, perm):
    """Procedura zyrtare fjalë për fjalë: skano permutacionin, shto brinjët {v,w} mes fqinjëve dalës."""
    pos = {v: i for i, v in enumerate(perm)}
    adj = {v: set() for v in range(g.n)}
    for u, v in g.edges:
        adj[u].add(v)
        adj[v].add(u)
    for u in perm:
        out = [v for v in adj[u] if pos[v] > pos[u]]
        for v, w in itertools.combinations(out, 2):
            if w not in adj[v]:
                adj[v].add(w)
                adj[w].add(v)
    return [sum(1 for v in adj[u] if pos[v] > pos[u]) for u in perm]


def random_graph(n, p, rng):
    edges = [(i, i + 1) for i in range(n - 1)]          # i lidhur
    edges += [(u, v) for u in range(n) for v in range(u + 2, n) if rng.random() < p]
    return Graph(n, edges)


class Tests(unittest.TestCase):
    def test_readme_example(self):
        g = Graph(10, README_EDGES)
        self.assertEqual(evaluate(g, [0, 1, 2, 4, 5, 3, 6, 8, 9, 7, 6]), (4, 2))

    def test_fill_matches_literal_procedure(self):
        rng = random.Random(0)
        for _ in range(200):
            n = rng.randint(2, 25)
            g = random_graph(n, rng.choice([0.05, 0.15, 0.3]), rng)
            perm = list(range(n))
            rng.shuffle(perm)
            self.assertEqual(outdegrees(g, perm), literal_outdegrees(g, perm))

    def test_hypervolume_and_subset(self):
        w = [5, 5, 4, 4, 3, 2, 2, 1, 0, 0]
        pts = front_from_widths(w)
        self.assertEqual(pts, [(0, 5), (2, 4), (4, 3), (5, 2), (7, 1), (8, 0)])
        self.assertEqual(hypervolume(pts, 10), staircase_hv(w))
        # nën-bashkësia optimale me DP = maksimumi me forcë brutale
        for k in (1, 2, 3, 4):
            best = max(hypervolume(list(c), 10) for c in itertools.combinations(pts, k))
            self.assertEqual(hypervolume(select_best_subset(pts, 10, k), 10), best)

    def test_suffix_widths_monotone(self):
        g = random_graph(30, 0.2, random.Random(1))
        w = suffix_widths(outdegrees(g, list(range(30))))
        self.assertTrue(all(a >= b for a, b in zip(w, w[1:])))
        self.assertEqual(w[-1], 0)


if __name__ == "__main__":
    unittest.main()

In [ ]:
!python -m unittest test_torso -v

### Ekzekutimi

In [ ]:
!python run.py --seconds 900 --seeds 2 --data data --out .

### Rezultatet

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open("results/results.md", encoding="utf-8").read().split("![")[0]))
display(Image("results/convergence.png"))
display(Image("results/pareto_fronts.png"))